# RFSoC 4×2 雙板即時攝影機 · 4 子載波 16-QAM
A 板將 USB 攝影機的 640 × 480 灰階 JPEG 裝入 307,200-byte BRAM 區塊。影像走 A 的 DAC_A → B 的 ADC_A，回覆走 B 的 DAC_A → A 的 ADC_A。兩板使用同一份 overlay。使用同軸時，依板卡規格在兩條路徑各配置適當衰減。

1. 把同一份部署包解壓到兩板各自的 Jupyter 資料夾，從該目錄開啟 notebook。攝影機接 A 板。
2. B 板用預設的 `ROLE = "receive"` 執行所有儲存格。看到 `RX_WAITING_FOR_PILOT` 再於 A 板設定 `ROLE = "send"` 並執行。
3. 錄影時間由你決定。要正常結束，**在 A 板按 Jupyter 的 ■「Interrupt Kernel」**。A 會收尾並送出結束封包，B 會自動完成 MP4。B 也可單獨中斷並保留已寫入的畫面。如兩板分別中斷，下次串流請依上述順序重新執行。

In [ ]:
from pathlib import Path
from webcam_live import run_live
from webcam_benchmark import run_single_board
from webcam_latency import show_latency

ROLE = "receive"                 # B: "receive"；A: "send"
BITFILE = Path("ofdm_qam16.bit")
CAMERA = 0                       # 僅 A 使用。也可填影片路徑，播完即結束
FPS = 30.0                       # A 的擷取目標幀率，B 會讀取 A 送來的設定
AMPLITUDE = 4096
LIVE_PREVIEW = True
RECORDING_CODEC = "auto"         # B 優先用 MJPEG 直接封裝 MP4，可改 "mp4v" 以重新編碼
OUTPUT_ROOT = Path("received_live_webcam")

MODE = "live"       # single_board 單板計時，live 雙板收發計時
BENCH_BLOCKS = 40
BENCH_FRAMES = 6             # 每個測試區塊的目標 JPEG 張數
BENCH_CAMERA = CAMERA        # 設 None 使用固定測試圖，可在沒有相機時執行

### 執行測試
**single_board** 只需一塊 RFSoC。測試會鎖定板卡、讓數據機保持 reset，量攝影機與 JPEG 編碼，再用這批 JPEG 填入真實 BRAM，執行接收、解碼及 MP4 寫入。測試結束會還原暫存資料。若沒有相機，將 `BENCH_CAMERA = None`。測試固定每區塊張數，用於可重複的比較。

**live** 依上面的 B、A 順序啟動；停止後會各自保存量測。A 的 RF 完成時間涵蓋硬體傳輸、ACK、重傳與軟體輪詢；B 等待 DONE 還包含等候 A 的空檔。

兩種模式都使用相同的 BRAM 短讀與背景錄影。B 驗證 300 個封包的品質、讀到區塊 SHA-256 結尾，釋放 BRAM，核對 SHA-256 後交出最新 JPEG。背景執行緒依 A 的時間戳寫 MP4。錄影佇列最多等候兩個區塊，滿載時接收端會等待，保留已驗證的錄影幀。

In [ ]:
if MODE == "single_board":
    summary = run_single_board(BITFILE, OUTPUT_ROOT, camera=BENCH_CAMERA, fps=FPS,
                               blocks=BENCH_BLOCKS, frames=BENCH_FRAMES, recording_codec=RECORDING_CODEC)
elif MODE == "live":
    summary = run_live(ROLE, BITFILE, OUTPUT_ROOT, camera=CAMERA, fps=FPS,
                       amplitude=AMPLITUDE, show_preview=LIVE_PREVIEW, recording_codec=RECORDING_CODEC)
else:
    raise ValueError("MODE must be single_board or live")
summary

### 如何看延遲
各階段列出樣本單位、平均、P50、P95 與最大值。Camera、TX、RX 和 Recorder 各自計算執行緒內的工作占比。單板 Trial 會在同一區塊用舊版完整讀法做逐 byte 對照。

雙板先看 B 的 `done_to_release`（可收下一區塊）、`rx_service`（預覽就緒）及 `record_queue_wait`（錄影是否塞住）。B 的 `summary.json` 會記錄第一與最後一個有效資料區塊之間的 `delivered_fps`。再看 A 的 `rf`、`retransmissions` 和取像排隊時間。現有 bit 每區塊固定傳 300 個 1,024-byte 封包。各板單獨計時；瀏覽器實際顯示時間需要另外量測。

`RECORDING_CODEC = "auto"` 在此 RFSoC 上保留 JPEG 編碼並封裝為 MP4。無法載入原生函式時會使用 Python 對齊存取及 OpenCV 的 mp4v。若播放器需要 MPEG-4 Part 2，B 可指定 `"mp4v"`，錄影同樣在背景執行。摘要會列出實際使用的編碼。

In [ ]:
show_latency(summary["latency_report"])
if MODE == "single_board":
    print(f'每區塊 {summary["frames_per_block"]} 幀，平均使用 {summary["mean_slot_used_pct"]:.1f}% 的 BRAM 空間')

每次執行都有獨立結果目錄。B 的 `live_received.mp4` 與 `summary.json` 位於 `received_live_webcam/<時間>_receive/`，摘要包含接收張數、捨棄區塊數和是否收到結束封包。A 端摘要記錄送出張數和佇列丟幀數。

若要用手邊的雙板 32 子載波版本做對照，可將原本的 `ofdm_air8_rfsoc4x2.zip` 解到子目錄，兩板的 `BITFILE` 同時改指向該處的 `ofdm_air8.bit`。程式會從該包的 manifest 核對硬體版本，並顯示其 QPSK/BPSK 設定。原本單板 Method B 的 `ofdm_rf.bit` 是本版的攝影機流程參考。

### 優化前的 RFSoC 4×2 單板基準（2026-11-25）

同一批 USB 攝影機 JPEG、640 × 480 灰階，每種設定測 20 個資料區塊。下表為本板實測；接收處理包含完整 BRAM 讀取、解包、JPEG 解碼和同步 MP4 寫入，計至可交給預覽的時間點。本測試由 PS 注入區塊，沒有 RF 對板。

| 每區塊幀數 | 有效空間 | 接收處理平均 / P95（ms/資料區塊） | 完整 BRAM 讀取（ms/區塊） | 實驗短讀（ms/區塊） |
|---:|---:|---:|---:|---:|
| 1 | 8.0% | 168.6 / 170.6 | 134.8 | 9.8 |
| 6 | 47.7% | 336.9 / 343.3 | 134.4 | 57.3 |
| 11 | 87.4% | 505.7 / 510.4 | 134.8 | 104.4 |

讀取平均含最後一個 EOS。完整讀取每次都掃過 307,200 bytes，因此小區塊的優先改善項是減少 MMIO 讀取量。另一輪每區塊 6 幀、40 個資料區塊的量測中，MP4 寫入平均 29.6 ms/輸出幀，占接收端處理時間 52.2%；完整讀取占 40.5%。

### 同一批攝影機影格的單板對照
在同一台 RFSoC 4×2 擷取 60 張 USB 攝影機 JPEG，再以原順序重播四次。每區塊 6 幀，40 個資料區塊加 EOS，平均使用 47.2% 的 BRAM。下表使用真實 BRAM，由 PS 注入資料。時間單位為 ms。

| 階段 | 舊版平均 | 快速版平均 | 舊版 P95 | 快速版 P95 |
|---|---:|---:|---:|---:|
| DONE → 預覽就緒／資料區塊 | 359.50 | 18.30 | 420.87 | 19.58 |
| 讀取 BRAM／區塊，含 EOS | 134.83 | 16.76 | 135.57 | 18.44 |
| MP4 寫入／輸出幀 | 33.33 | 5.20 | 52.67 | 5.97 |

快速版從 DONE 到釋放 BRAM 平均 17.38 ms，背景錄影入列平均等候 0.10 ms。相同資料改用背景 mp4v 時，預覽就緒仍為 18.57 ms，入列等候升到 66.91 ms。三次 MP4 均可完整解碼為 240 幀、30 fps。這 8 秒的 MJPEG MP4 為 5.79 MB，mp4v 為 1.99 MB。雙板上的實際 RF 重傳、有效接收幀率與 Jupyter 顯示請用 live 模式量測。